In [1]:
!pip install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel

spark = SparkSession.builder.appName("Assessment1").getOrCreate()

In [3]:
uploaded = files.upload()

Saving tariff_master.csv to tariff_master.csv
Saving substation_master.csv to substation_master.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_master.csv to meter_master.csv
Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv


In [4]:
def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

readings_df = load_csv("meter_readings_90000.csv")
extra_df = load_csv("meter_readings_extra_12000.csv")
meter_df = load_csv("meter_master.csv")
substation_df = load_csv("substation_master.csv")
tariff_df = load_csv("tariff_master.csv")
tickets_df = load_csv("maintenance_tickets_1600.csv")


A. Inspect, select and transform

In [6]:
readings_df.printSchema()

print("Row count:", readings_df.count())

readings_df.show(10)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Row count: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04

In [8]:
selected_df = readings_df.select(
    "reading_id", "meter_id", "reading_timestamp", "energy_kwh", "reading_status"
)

selected_df.show()

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
| RD0000006|   M0139|2026-02-20 02:15:00|      0.36|         Valid|
| RD0000007|   M0142|2026-02-01 05:30:00|     79.47|         Valid|
| RD0000008|   M0126|2026-03-01 20:45:00|     28.35|      Rejected|
| RD0000009|   M0005|2026-05-03 10:30:00|      1.96|     Estimated|
| RD0000010|   M0052|2026-04-10 05:45:00|     22.52|         Valid|
| RD0000011|   M0137|2026-02-03 16:00:00|     10.51|         Valid|
| RD0000012|   M0009|2026-05-01 00:00:00|      1

In [9]:
renamed_df = readings_df.withColumnRenamed("source_type", "collection_mode")

renamed_df.select("reading_id", "collection_mode").show()


+----------+---------------+
|reading_id|collection_mode|
+----------+---------------+
| RD0000001|     SmartMeter|
| RD0000002|     SmartMeter|
| RD0000003|     SmartMeter|
| RD0000004|         Manual|
| RD0000005|     SmartMeter|
| RD0000006|     SmartMeter|
| RD0000007|     SmartMeter|
| RD0000008|     SmartMeter|
| RD0000009|     SmartMeter|
| RD0000010|     SmartMeter|
| RD0000011|     SmartMeter|
| RD0000012|     SmartMeter|
| RD0000013|     SmartMeter|
| RD0000014|     SmartMeter|
| RD0000015|     SmartMeter|
| RD0000016|     SmartMeter|
| RD0000017|     SmartMeter|
| RD0000018|     SmartMeter|
| RD0000019|     SmartMeter|
| RD0000020|     SmartMeter|
+----------+---------------+
only showing top 20 rows


In [11]:
high_valid_df = readings_df \
    .filter((F.col("reading_status") == "Valid") & (F.col("energy_kwh") > 20)) \
    .orderBy(F.col("energy_kwh").desc())

high_valid_df.show()

high_valid_df.count()

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             0|         Valid| SmartMeter|                 25.3|      OP017|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             

24451

In [12]:
band_df = readings_df.withColumn(
    "energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High")
)

band_df.groupBy("energy_band").count().orderBy(F.col("count").desc()).show()

+-----------+-----+
|energy_band|count|
+-----------+-----+
|     Medium|39731|
|        Low|26898|
|       High|21751|
|    Unknown| 1620|
+-----------+-----+



B. Null values and data cleaning

In [13]:
readings_df.select(
    F.count(F.when(F.col("energy_kwh").isNull(), 1)).alias("null_energy_kwh"),
    F.count(F.when(F.col("voltage_v").isNull(), 1)).alias("null_voltage_v"),
    F.count(F.when(F.col("power_factor").isNull(), 1)).alias("null_power_factor"),
    F.count(F.when(F.col("operator_id").isNull(), 1)).alias("null_operator_id")
).show()

+---------------+--------------+-----------------+----------------+
|null_energy_kwh|null_voltage_v|null_power_factor|null_operator_id|
+---------------+--------------+-----------------+----------------+
|           1620|          2070|             1440|            5400|
+---------------+--------------+-----------------+----------------+



In [14]:
filled_df = readings_df.na.fill({"operator_id": "UNASSIGNED", "power_factor": 0.90})

filled_df.select(
    F.count(F.when(F.col("operator_id").isNull(), 1)).alias("null_operator_id_new"),
    F.count(F.when(F.col("power_factor").isNull(), 1)).alias("null_power_factor_new")
).show()

readings_df.select(
    F.count(F.when(F.col("operator_id").isNull(), 1)).alias("null_operator_id_original"),
    F.count(F.when(F.col("power_factor").isNull(), 1)).alias("null_power_factor_original")
).show()

+--------------------+---------------------+
|null_operator_id_new|null_power_factor_new|
+--------------------+---------------------+
|                   0|                    0|
+--------------------+---------------------+

+-------------------------+--------------------------+
|null_operator_id_original|null_power_factor_original|
+-------------------------+--------------------------+
|                     5400|                      1440|
+-------------------------+--------------------------+



In [15]:
before_count = readings_df.count()

clean_df = readings_df.na.drop(subset=["energy_kwh", "voltage_v"])

after_count = clean_df.count()

print("Before:", before_count)
print("After:", after_count)
print("Rows removed:", before_count - after_count)

Before: 90000
After: 86349
Rows removed: 3651


C. Joins, groupBy and aggregations

In [16]:
readings_df.join(meter_df, on="meter_id", how="inner") \
    .filter(F.col("reading_status") == "Valid") \
    .groupBy("site_type") \
    .agg(F.round(F.sum("energy_kwh"), 2).alias("total_valid_energy_kwh")) \
    .orderBy(F.col("total_valid_energy_kwh").desc()) \
    .show()

+---------------+----------------------+
|      site_type|total_valid_energy_kwh|
+---------------+----------------------+
|     Industrial|             710317.41|
|     Commercial|             451525.18|
|Public Facility|             138870.41|
|    Residential|             100863.29|
+---------------+----------------------+



In [17]:
readings_df.join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .filter(F.col("reading_status") == "Valid") \
    .groupBy("region", "city") \
    .agg(F.round(F.sum("energy_kwh"), 2).alias("total_valid_energy_kwh")) \
    .orderBy("region", "city") \
    .show()

+-------+-------------+----------------------+
| region|         city|total_valid_energy_kwh|
+-------+-------------+----------------------+
|Central|       Indore|             176503.91|
|  North|        Delhi|              94234.89|
|  North|       Jaipur|              47863.16|
|  North|      Lucknow|              142610.4|
|  South|    Bengaluru|               73787.6|
|  South|      Chennai|              51829.06|
|  South|    Hyderabad|              51937.79|
|  South|        Kochi|             204272.46|
|  South|Visakhapatnam|             205015.83|
|   West|    Ahmedabad|             138627.74|
|   West|       Mumbai|              52116.21|
|   West|         Pune|             162777.24|
+-------+-------------+----------------------+



In [19]:
charge_df = readings_df \
    .join(meter_df, on="meter_id", how="inner") \
    .join(tariff_df, on="tariff_id", how="inner") \
    .filter((F.col("reading_status") != "Rejected") & F.col("energy_kwh").isNotNull()) \
    .withColumn("energy_charge_inr", F.col("energy_kwh") * F.col("rate_per_kwh_inr"))

charge_df.select(
    "reading_id", "meter_id", "tariff_id", "energy_kwh", "rate_per_kwh_inr", "energy_charge_inr"
).show()

charge_df.agg(
    F.count("*").alias("readings_charged"),
    F.round(F.sum("energy_charge_inr"), 2).alias("total_energy_charge_inr")
).show()

+----------+--------+-------------------+----------+----------------+------------------+
|reading_id|meter_id|          tariff_id|energy_kwh|rate_per_kwh_inr| energy_charge_inr|
+----------+--------+-------------------+----------+----------------+------------------+
| RD0000001|   M0082|2026-10-30 05:00:00|     22.53|             6.7|150.95100000000002|
| RD0000002|   M0020|2026-10-30 02:00:00|      2.97|             6.4|19.008000000000003|
| RD0000003|   M0054|2026-10-30 03:00:00|      8.61|             8.2| 70.60199999999999|
| RD0000004|   M0111|2026-10-30 01:00:00|      1.87|             5.1| 9.536999999999999|
| RD0000005|   M0135|2026-10-30 03:00:00|      4.39|             8.2|            35.998|
| RD0000006|   M0139|2026-10-30 02:00:00|      0.36|             6.4|             2.304|
| RD0000007|   M0142|2026-10-30 04:00:00|     79.47|             9.1| 723.1769999999999|
| RD0000009|   M0005|2026-10-30 01:00:00|      1.96|             5.1| 9.995999999999999|
| RD0000010|   M0052|

In [21]:
charge_df.groupBy("tariff_id", "tariff_name") \
    .agg(F.round(F.sum("energy_charge_inr"), 2).alias("total_energy_charge_inr")) \
    .orderBy("tariff_id") \
    .show()

+-------------------+--------------------+-----------------------+
|          tariff_id|         tariff_name|total_energy_charge_inr|
+-------------------+--------------------+-----------------------+
|2026-10-30 01:00:00|Residential Standard|              220551.29|
|2026-10-30 02:00:00|    Residential Peak|              473336.45|
|2026-10-30 03:00:00|       Business Flex|             4288997.78|
|2026-10-30 04:00:00|    Industrial Power|             7487737.98|
|2026-10-30 05:00:00|     Public Services|             1077282.75|
+-------------------+--------------------+-----------------------+



In [23]:
tickets_df.join(meter_df, on="meter_id", how="inner") \
    .groupBy("site_type", "severity") \
    .agg(
        F.count("*").alias("ticket_count"),
        F.round(F.avg("maintenance_cost_inr"), 2).alias("avg_maintenance_cost_inr")
    ) \
    .orderBy("site_type", "severity") \
    .show()

+---------------+--------+------------+------------------------+
|      site_type|severity|ticket_count|avg_maintenance_cost_inr|
+---------------+--------+------------+------------------------+
|     Commercial|Critical|          37|                13424.73|
|     Commercial|    High|         130|                12293.07|
|     Commercial|     Low|         166|                12223.52|
|     Commercial|  Medium|         171|                 13706.6|
|     Industrial|Critical|          17|                 12296.2|
|     Industrial|    High|          63|                12838.87|
|     Industrial|     Low|          66|                12717.53|
|     Industrial|  Medium|         106|                 12043.9|
|Public Facility|Critical|          15|                 8986.84|
|Public Facility|    High|          48|                12905.28|
|Public Facility|     Low|          60|                11195.93|
|Public Facility|  Medium|          97|                13050.63|
|    Residential|Critical

In [25]:
readings_df.join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .groupBy("substation_id", "city") \
    .agg(
        F.round(F.avg("voltage_v"), 2).alias("avg_voltage_v"),
        F.sum("outage_minutes").alias("total_outage_minutes")
    ) \
    .orderBy(F.col("total_outage_minutes").desc()) \
    .show()

+-------------+-------------+-------------+--------------------+
|substation_id|         city|avg_voltage_v|total_outage_minutes|
+-------------+-------------+-------------+--------------------+
|        SUB10|Visakhapatnam|       230.04|               91224|
|        SUB09|        Kochi|       230.05|               81734|
|        SUB04|         Pune|       230.06|               75872|
|        SUB08|    Ahmedabad|       229.88|               75581|
|        SUB03|      Chennai|       230.03|               61662|
|        SUB12|       Indore|       230.14|               60290|
|        SUB05|       Mumbai|       230.02|               54211|
|        SUB01|    Hyderabad|       229.78|               48172|
|        SUB11|      Lucknow|       230.03|               45720|
|        SUB06|        Delhi|       230.06|               44165|
|        SUB02|    Bengaluru|       229.89|               38857|
|        SUB07|       Jaipur|       229.87|               30218|
+-------------+----------

In [26]:
readings_df.groupBy("source_type", "reading_status") \
    .agg(F.count("*").alias("reading_count")) \
    .orderBy(F.col("reading_count").desc()) \
    .show()

+-----------+--------------+-------------+
|source_type|reading_status|reading_count|
+-----------+--------------+-------------+
| SmartMeter|         Valid|        65521|
| SmartMeter|     Estimated|        10443|
|     Manual|         Valid|         8965|
| SmartMeter|      Rejected|         3202|
|     Manual|     Estimated|         1405|
|     Manual|      Rejected|          464|
+-----------+--------------+-------------+



In [27]:
readings_df.groupBy("source_type") \
    .pivot("reading_status", ["Valid", "Estimated", "Rejected"]) \
    .count() \
    .show()

+-----------+-----+---------+--------+
|source_type|Valid|Estimated|Rejected|
+-----------+-----+---------+--------+
|     Manual| 8965|     1405|     464|
| SmartMeter|65521|    10443|    3202|
+-----------+-----+---------+--------+



D. Date and window functions

In [28]:
import time
import os

In [30]:
timestamp_df = readings_df \
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("year", F.year("reading_timestamp")) \
    .withColumn("month", F.month("reading_timestamp"))

timestamp_df.select("reading_id", "reading_timestamp", "year", "month").show()

+----------+-------------------+----+-----+
|reading_id|  reading_timestamp|year|month|
+----------+-------------------+----+-----+
| RD0000001|2026-04-01 02:00:00|2026|    4|
| RD0000002|2026-01-31 06:00:00|2026|    1|
| RD0000003|2026-04-11 10:15:00|2026|    4|
| RD0000004|2026-03-14 00:00:00|2026|    3|
| RD0000005|2026-05-27 01:45:00|2026|    5|
| RD0000006|2026-02-20 02:15:00|2026|    2|
| RD0000007|2026-02-01 05:30:00|2026|    2|
| RD0000008|2026-03-01 20:45:00|2026|    3|
| RD0000009|2026-05-03 10:30:00|2026|    5|
| RD0000010|2026-04-10 05:45:00|2026|    4|
| RD0000011|2026-02-03 16:00:00|2026|    2|
| RD0000012|2026-05-01 00:00:00|2026|    5|
| RD0000013|2026-02-01 16:15:00|2026|    2|
| RD0000014|2026-05-17 03:00:00|2026|    5|
| RD0000015|2026-01-17 22:00:00|2026|    1|
| RD0000016|2026-04-16 18:00:00|2026|    4|
| RD0000017|2026-06-14 21:15:00|2026|    6|
| RD0000018|2026-05-04 07:45:00|2026|    5|
| RD0000019|2026-03-21 16:45:00|2026|    3|
| RD0000020|2026-01-14 13:00:00|

In [32]:
timestamp_df \
    .filter(F.col("reading_status") == "Valid") \
    .join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .groupBy("region", "year", "month") \
    .agg(F.round(F.sum("energy_kwh"), 2).alias("valid_energy_kwh")) \
    .orderBy("region", "year", "month") \
    .show()

+-------+----+-----+----------------+
| region|year|month|valid_energy_kwh|
+-------+----+-----+----------------+
|Central|2026|    1|        31475.58|
|Central|2026|    2|        28161.41|
|Central|2026|    3|        29805.85|
|Central|2026|    4|        29475.34|
|Central|2026|    5|        30048.88|
|Central|2026|    6|        27536.85|
|  North|2026|    1|        48849.31|
|  North|2026|    2|        45444.25|
|  North|2026|    3|        51094.61|
|  North|2026|    4|        43451.02|
|  North|2026|    5|        48796.93|
|  North|2026|    6|        47072.33|
|  South|2026|    1|        97547.39|
|  South|2026|    2|         90864.8|
|  South|2026|    3|       102258.77|
|  South|2026|    4|        97266.94|
|  South|2026|    5|       101204.87|
|  South|2026|    6|        97699.97|
|   West|2026|    1|        59967.57|
|   West|2026|    2|         55807.9|
+-------+----+-----+----------------+
only showing top 20 rows


In [33]:
meter_totals_df = timestamp_df \
    .filter(F.col("reading_status") == "Valid") \
    .join(meter_df, on="meter_id", how="inner") \
    .groupBy("substation_id", "meter_id") \
    .agg(F.sum("energy_kwh").alias("total_valid_energy_kwh"))

meter_rank_window = Window.partitionBy("substation_id") \
    .orderBy(F.col("total_valid_energy_kwh").desc())

meter_totals_df \
    .withColumn("meter_rank", F.dense_rank().over(meter_rank_window)) \
    .filter(F.col("meter_rank") <= 3) \
    .select(
        "substation_id", "meter_id",
        F.round("total_valid_energy_kwh", 2).alias("total_valid_energy_kwh"),
        "meter_rank"
    ) \
    .orderBy("substation_id", "meter_rank") \
    .show()

+-------------+--------+----------------------+----------+
|substation_id|meter_id|total_valid_energy_kwh|meter_rank|
+-------------+--------+----------------------+----------+
|        SUB01|   M0130|               9364.98|         1|
|        SUB01|   M0110|               9259.75|         2|
|        SUB01|   M0068|               8832.58|         3|
|        SUB02|   M0071|              29242.04|         1|
|        SUB02|   M0145|              28641.32|         2|
|        SUB02|   M0133|               7060.14|         3|
|        SUB03|   M0096|               9632.74|         1|
|        SUB03|   M0080|               9525.94|         2|
|        SUB03|   M0073|               9369.79|         3|
|        SUB04|   M0038|              29505.88|         1|
|        SUB04|   M0099|              29216.32|         2|
|        SUB04|   M0098|              28613.46|         3|
|        SUB05|   M0094|               9659.64|         1|
|        SUB05|   M0050|               9570.86|         

In [34]:
cumulative_window = Window.partitionBy("meter_id") \
    .orderBy("reading_timestamp", "reading_id") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

cumulative_df = timestamp_df \
    .filter((F.col("reading_status") == "Valid") & F.col("energy_kwh").isNotNull()) \
    .withColumn(
        "cumulative_valid_energy_kwh",
        F.round(F.sum("energy_kwh").over(cumulative_window), 2)
    )

cumulative_df.filter(F.col("meter_id") == "M0001") \
    .select("meter_id", "reading_timestamp", "reading_id", "energy_kwh", "cumulative_valid_energy_kwh") \
    .orderBy("reading_timestamp", "reading_id") \
    .show()

+--------+-------------------+----------+----------+---------------------------+
|meter_id|  reading_timestamp|reading_id|energy_kwh|cumulative_valid_energy_kwh|
+--------+-------------------+----------+----------+---------------------------+
|   M0001|2026-01-01 01:15:00| RD0083587|     22.27|                      22.27|
|   M0001|2026-01-01 03:30:00| RD0086275|      4.05|                      26.32|
|   M0001|2026-01-01 09:45:00| RD0088204|     11.28|                       37.6|
|   M0001|2026-01-02 04:45:00| RD0005149|     11.76|                      49.36|
|   M0001|2026-01-02 08:15:00| RD0035530|     10.06|                      59.42|
|   M0001|2026-01-02 18:30:00| RD0070104|     15.83|                      75.25|
|   M0001|2026-01-03 00:30:00| RD0006677|     17.06|                      92.31|
|   M0001|2026-01-03 06:00:00| RD0021042|     17.14|                     109.45|
|   M0001|2026-01-03 08:30:00| RD0032205|     20.46|                     129.91|
|   M0001|2026-01-03 18:15:0

E. Union and deduplication

In [35]:
combined_df = readings_df.unionByName(extra_df)

print("Main file:", readings_df.count())
print("Extra file:", extra_df.count())
print("Combined:", combined_df.count())

Main file: 90000
Extra file: 12000
Combined: 102000


In [36]:
repeated_df = combined_df.groupBy("reading_id") \
    .count() \
    .filter(F.col("count") > 1)

print("Repeated reading_id values:", repeated_df.count())

repeated_df.orderBy("reading_id").show()

Repeated reading_id values: 700
+----------+-----+
|reading_id|count|
+----------+-----+
| RD0000004|    2|
| RD0000121|    2|
| RD0000194|    2|
| RD0000545|    2|
| RD0000650|    2|
| RD0000715|    2|
| RD0000765|    2|
| RD0001057|    2|
| RD0001183|    2|
| RD0001235|    2|
+----------+-----+
only showing top 10 rows


In [37]:
unique_df = combined_df.dropDuplicates(["reading_id"])

print("Before deduplication:", combined_df.count())
print("After deduplication:", unique_df.count())
print("Rows removed:", combined_df.count() - unique_df.count())
print("Distinct reading_id values:", combined_df.select("reading_id").distinct().count())

Before deduplication: 102000
After deduplication: 101300
Rows removed: 700
Distinct reading_id values: 101300


F. Lazy execution, caching and persistence

In [38]:
pipeline_df = readings_df \
    .filter(F.col("reading_status") == "Valid") \
    .withColumn("high_load_flag", F.when(F.col("energy_kwh") > 25, 1).otherwise(0)) \
    .groupBy("source_type") \
    .agg(
        F.count("*").alias("valid_readings"),
        F.round(F.sum("energy_kwh"), 2).alias("total_energy_kwh"),
        F.sum("high_load_flag").alias("high_load_readings")
    ) \
    .orderBy(F.col("total_energy_kwh").desc())

pipeline_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_energy_kwh#1693 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_energy_kwh#1693 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=4806]
      +- HashAggregate(keys=[source_type#25], functions=[count(1), sum(energy_kwh#20), sum(high_load_flag#1691)])
         +- Exchange hashpartitioning(source_type#25, 200), ENSURE_REQUIREMENTS, [plan_id=4803]
            +- HashAggregate(keys=[source_type#25], functions=[partial_count(1), partial_sum(energy_kwh#20), partial_sum(high_load_flag#1691)])
               +- Project [energy_kwh#20, source_type#25, CASE WHEN (energy_kwh#20 > 25.0) THEN 1 ELSE 0 END AS high_load_flag#1691]
                  +- Filter (isnotnull(reading_status#24) AND (reading_status#24 = Valid))
                     +- FileScan csv [energy_kwh#20,reading_status#24,source_type#25] Batched: false, DataFilters: [isnotnull(reading_status#24), (reading_status#24 = Valid)], Format: CSV, Locati

In [39]:
pipeline_df.show()

+-----------+--------------+----------------+------------------+
|source_type|valid_readings|total_energy_kwh|high_load_readings|
+-----------+--------------+----------------+------------------+
| SmartMeter|         65521|      1234716.34|             15930|
|     Manual|          8965|       166859.95|              2135|
+-----------+--------------+----------------+------------------+



In [40]:
valid_df = readings_df.filter(F.col("reading_status") == "Valid")

valid_df.cache()

start = time.time()
first_count = valid_df.count()
first_time = time.time() - start

start = time.time()
second_count = valid_df.count()
second_time = time.time() - start

print("First count:", first_count, "-", round(first_time, 4), "seconds")
print("Second count:", second_count, "-", round(second_time, 4), "seconds")
print("Is cached:", valid_df.is_cached)

First count: 74486 - 1.7105 seconds
Second count: 74486 - 0.2537 seconds
Is cached: True


In [41]:
outage_df = readings_df.filter(F.col("outage_minutes") > 0)

outage_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Readings with outage:", outage_df.count())

outage_df.groupBy("reading_status") \
    .agg(F.count("*").alias("readings"), F.sum("outage_minutes").alias("total_outage_minutes")) \
    .orderBy(F.col("readings").desc()) \
    .show()

print(outage_df.storageLevel)

Readings with outage: 7653
+--------------+--------+--------------------+
|reading_status|readings|total_outage_minutes|
+--------------+--------+--------------------+
|         Valid|    6345|              589369|
|     Estimated|    1000|               89057|
|      Rejected|     308|               29280|
+--------------+--------+--------------------+

Disk Memory Serialized 1x Replicated


In [42]:
outage_df.unpersist(blocking=True)

print("Is cached:", outage_df.is_cached)

Is cached: False


In [43]:
def time_runs(base_df, runs=3):
    times = []
    for i in range(runs):
        start = time.time()
        base_df.groupBy("meter_id") \
            .agg(F.sum("energy_kwh").alias("total_energy_kwh")) \
            .orderBy(F.col("total_energy_kwh").desc()) \
            .collect()
        times.append(round(time.time() - start, 4))
    return times

uncached_base_df = readings_df.filter(F.col("reading_status") == "Valid")

uncached_times = time_runs(uncached_base_df)

print("Without caching (seconds):", uncached_times)

Without caching (seconds): [1.0279, 0.3448, 0.8395]


In [44]:
cached_base_df = readings_df.filter(F.col("reading_status") == "Valid")

cached_base_df.cache()

cached_base_df.count()

cached_times = time_runs(cached_base_df)

print("With caching (seconds):", cached_times)

With caching (seconds): [0.5052, 0.3123, 0.3562]


In [45]:
print("Without caching:", uncached_times, "average", round(sum(uncached_times) / len(uncached_times), 4))
print("With caching:   ", cached_times, "average", round(sum(cached_times) / len(cached_times), 4))

valid_df.unpersist(blocking=True)
cached_base_df.unpersist(blocking=True)

Without caching: [1.0279, 0.3448, 0.8395] average 0.7374
With caching:    [0.5052, 0.3123, 0.3562] average 0.3912


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

G. Partitions and output

In [46]:
readings_8_df = readings_df.repartition(8)

print("Partitions:", readings_8_df.rdd.getNumPartitions())

readings_8_df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11249|
|           1|11249|
|           2|11251|
|           3|11251|
|           4|11250|
|           5|11250|
|           6|11250|
|           7|11250|
+------------+-----+



In [47]:
readings_12_df = readings_df.repartition(12)

coalesced_3_df = readings_12_df.coalesce(3)
repartitioned_3_df = readings_12_df.repartition(3)

print("Initial:", readings_12_df.rdd.getNumPartitions())
print("Coalesce(3):", coalesced_3_df.rdd.getNumPartitions())
print("Repartition(3):", repartitioned_3_df.rdd.getNumPartitions())

Initial: 12
Coalesce(3): 3
Repartition(3): 3


In [48]:
print("COALESCE PLAN")
coalesced_3_df.explain()

print("REPARTITION PLAN")
repartitioned_3_df.explain()

COALESCE PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=5918]
            +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...
+- == Initial Plan ==
   Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=5918]
      +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_

In [49]:
enriched_df = readings_df \
    .na.drop(subset=["energy_kwh", "voltage_v"]) \
    .join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner")

enriched_df.repartition(4) \
    .write.mode("overwrite") \
    .parquet("enriched_readings_parquet")

In [50]:
for name in sorted(os.listdir("enriched_readings_parquet")):
    path = os.path.join("enriched_readings_parquet", name)
    print(name, "-", os.path.getsize(path), "bytes")

spark.read.parquet("enriched_readings_parquet").count()


._SUCCESS.crc - 8 bytes
.part-00000-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet.crc - 4140 bytes
.part-00001-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet.crc - 4140 bytes
.part-00002-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet.crc - 4140 bytes
.part-00003-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet.crc - 4144 bytes
_SUCCESS - 0 bytes
part-00000-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet - 528684 bytes
part-00001-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet - 528680 bytes
part-00002-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet - 528774 bytes
part-00003-2dafd59a-d336-41be-8a9b-8d9494d192b8-c000.snappy.parquet - 528911 bytes


86349

Final Capstone


In [51]:
combined_df = readings_df.unionByName(extra_df)

unique_readings_df = combined_df.dropDuplicates(["reading_id"])

print("Main file:", readings_df.count())
print("Extra file:", extra_df.count())
print("Combined:", combined_df.count())
print("Unique after deduplication:", unique_readings_df.count())

Main file: 90000
Extra file: 12000
Combined: 102000
Unique after deduplication: 101300


In [52]:
clean_df = unique_readings_df \
    .withColumn(
        "is_billable",
        F.when((F.col("reading_status") != "Rejected") & F.col("energy_kwh").isNotNull(), 1).otherwise(0)
    ) \
    .withColumn(
        "billable_kwh",
        F.when(F.col("is_billable") == 1, F.col("energy_kwh"))
    )

clean_df.select(
    F.count("*").alias("total_readings"),
    F.count(F.when(F.col("energy_kwh").isNull(), 1)).alias("null_energy_readings"),
    F.count(F.when(F.col("reading_status") == "Rejected", 1)).alias("rejected_readings"),
    F.sum("is_billable").alias("billable_readings")
).show()

+--------------+--------------------+-----------------+-----------------+
|total_readings|null_energy_readings|rejected_readings|billable_readings|
+--------------+--------------------+-----------------+-----------------+
|        101300|                1823|             4136|            95408|
+--------------+--------------------+-----------------+-----------------+



In [53]:
joined_df = clean_df \
    .join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .join(tariff_df, on="tariff_id", how="inner")

print("Rows after joins:", joined_df.count())


Rows after joins: 101300


In [55]:
enriched_df = joined_df \
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("month", F.date_format("reading_timestamp", "yyyy-MM")) \
    .withColumn("energy_charge_inr", F.col("billable_kwh") * F.col("rate_per_kwh_inr")) \
    .withColumn("outage_flag", F.when(F.col("outage_minutes") > 0, 1).otherwise(0))

enriched_df.select(
    "reading_id", "month", "region", "site_type", "reading_status",
    "energy_kwh", "billable_kwh", "rate_per_kwh_inr", "energy_charge_inr", "outage_minutes"
).show()

+----------+-------+-------+---------------+--------------+----------+------------+----------------+------------------+--------------+
|reading_id|  month| region|      site_type|reading_status|energy_kwh|billable_kwh|rate_per_kwh_inr| energy_charge_inr|outage_minutes|
+----------+-------+-------+---------------+--------------+----------+------------+----------------+------------------+--------------+
| RD0000002|2026-01|   West|    Residential|         Valid|      2.97|        2.97|             6.4|19.008000000000003|             0|
| RD0000003|2026-04|Central|     Commercial|     Estimated|      8.61|        8.61|             8.2| 70.60199999999999|             0|
| RD0000004|2026-03|Central|    Residential|         Valid|      1.87|        1.87|             5.1| 9.536999999999999|             0|
| RD0000005|2026-05|Central|     Commercial|         Valid|      4.39|        4.39|             8.2|            35.998|             0|
| RD0000011|2026-02|  South|Public Facility|         Va

In [56]:
report_df = enriched_df \
    .groupBy("month", "region", "site_type") \
    .agg(
        F.count("*").alias("total_readings"),
        F.sum("is_billable").alias("billable_readings"),
        F.round(F.sum("billable_kwh"), 2).alias("total_billable_kwh"),
        F.round(F.sum("energy_charge_inr"), 2).alias("total_energy_charge_inr"),
        F.sum("outage_minutes").alias("total_outage_minutes"),
        F.sum("outage_flag").alias("outage_readings")
    ) \
    .withColumn(
        "outage_reading_pct",
        F.round(F.col("outage_readings") / F.col("total_readings") * 100, 2)
    )

In [57]:
rank_window = Window.partitionBy("region", "month") \
    .orderBy(F.col("total_energy_charge_inr").desc())

final_df = report_df.withColumn("charge_rank", F.dense_rank().over(rank_window))

final_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [dense_rank(total_energy_charge_inr#5719) windowspecdefinition(region#99, month#5602, total_energy_charge_inr#5719 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS charge_rank#5759], [region#99, month#5602], [total_energy_charge_inr#5719 DESC NULLS LAST]
   +- Sort [region#99 ASC NULLS FIRST, month#5602 ASC NULLS FIRST, total_energy_charge_inr#5719 DESC NULLS LAST], false, 0
      +- Exchange hashpartitioning(region#99, month#5602, 200), ENSURE_REQUIREMENTS, [plan_id=7802]
         +- Project [month#5602, region#99, site_type#75, total_readings#5716L, billable_readings#5717L, total_billable_kwh#5718, total_energy_charge_inr#5719, total_outage_minutes#5720L, outage_readings#5721L, round(((cast(outage_readings#5721L as double) / cast(total_readings#5716L as double)) * 100.0), 2) AS outage_reading_pct#5758]
            +- HashAggregate(keys=[month#5602, region#99, site_type#75], func

In [58]:
final_df.cache()

print("Report rows:", final_df.count())

final_df.filter(F.col("month") == "2026-01") \
    .orderBy("region", "charge_rank") \
    .show(20, truncate=False)

print("Is cached:", final_df.is_cached)

Report rows: 120
+-------+-------+---------------+--------------+-----------------+------------------+-----------------------+--------------------+---------------+------------------+-----------+
|month  |region |site_type      |total_readings|billable_readings|total_billable_kwh|total_energy_charge_inr|total_outage_minutes|outage_readings|outage_reading_pct|charge_rank|
+-------+-------+---------------+--------------+-----------------+------------------+-----------------------+--------------------+---------------+------------------+-----------+
|2026-01|Central|Industrial     |440           |418              |24871.22          |226328.1               |4396                |49             |11.14             |1          |
|2026-01|Central|Commercial     |595           |567              |10649.03          |87322.05               |5445                |62             |10.42             |2          |
|2026-01|Central|Residential    |234           |224              |814.03            |4636.91 

In [59]:
final_df.groupBy("month") \
    .agg(
        F.round(F.sum("total_billable_kwh"), 2).alias("total_billable_kwh"),
        F.round(F.sum("total_energy_charge_inr"), 2).alias("total_energy_charge_inr"),
        F.sum("total_outage_minutes").alias("total_outage_minutes")
    ) \
    .orderBy("month") \
    .show()

+-------+------------------+-----------------------+--------------------+
|  month|total_billable_kwh|total_energy_charge_inr|total_outage_minutes|
+-------+------------------+-----------------------+--------------------+
|2026-01|         276061.63|             2303606.03|              120590|
|2026-02|         253065.87|             2112375.19|              111526|
|2026-03|         282890.15|             2363237.65|              129560|
|2026-04|         265540.48|             2212041.62|              111961|
|2026-05|         280626.11|             2342561.92|              119834|
|2026-06|         265685.26|             2214083.79|              114235|
|2026-07|         103267.89|              861740.53|               40596|
|2026-08|         104289.18|              873049.71|               43137|
+-------+------------------+-----------------------+--------------------+



In [60]:
final_df.unpersist(blocking=True)

print("Is cached:", final_df.is_cached)

Is cached: False


In [61]:
final_df.repartition(2) \
    .write.mode("overwrite") \
    .parquet("energy_consumption_reliability_report")

In [63]:
for name in sorted(os.listdir("energy_consumption_reliability_report")):
    path = os.path.join("energy_consumption_reliability_report", name)
    print(name, "-", os.path.getsize(path), "bytes")

saved_df = spark.read.parquet("energy_consumption_reliability_report")

print("Rows read back:", saved_df.count())

saved_df.orderBy("month", "region", "charge_rank").show()

._SUCCESS.crc - 8 bytes
.part-00000-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet.crc - 56 bytes
.part-00001-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet.crc - 56 bytes
_SUCCESS - 0 bytes
part-00000-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet - 6005 bytes
part-00001-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet - 6000 bytes
Rows read back: 120
+-------+-------+---------------+--------------+-----------------+------------------+-----------------------+--------------------+---------------+------------------+-----------+
|  month| region|      site_type|total_readings|billable_readings|total_billable_kwh|total_energy_charge_inr|total_outage_minutes|outage_readings|outage_reading_pct|charge_rank|
+-------+-------+---------------+--------------+-----------------+------------------+-----------------------+--------------------+---------------+------------------+-----------+
|2026-01|Central|     Industrial|           440|              418|       

In [64]:
import shutil

shutil.make_archive("energy_consumption_reliability_report", "zip", "energy_consumption_reliability_report")

'/content/energy_consumption_reliability_report.zip'

In [65]:
import zipfile

with zipfile.ZipFile("energy_consumption_reliability_report.zip") as z:
    for name in z.namelist():
        print(name)

.part-00000-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet.crc
_SUCCESS
part-00000-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet
part-00001-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet
.part-00001-985a7f45-9a9d-4dfb-a71b-93c442a1f4b1-c000.snappy.parquet.crc
._SUCCESS.crc


In [66]:
from google.colab import files

files.download("energy_consumption_reliability_report.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>